# Final Tract-Level Merge

I'm combining the cleaned tract-level datasets into one master file.

The goal is one row per San Diego County census tract.

ACS is the base dataset because it has the full tract list. From there, I'll merge the other datasets using a cleaned `tract_id` column.

I’ll keep the full master file for now, and try to avoid dupe columns. Once everything is merged, I'll make a smaller scoring dataset with only the features I actually need.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 150)
pd.set_option('display.max_rows', 100)

# local project path
project_path = Path.cwd().parent

data_path = project_path / 'data'
processed_path = data_path / 'processed'

paths = {
    'acs': processed_path / 'acs_tracts_selected_2024.csv',
    'crime': processed_path / 'crime_safety_by_tract.csv',
    'walkability': processed_path / 'walkability_by_tract.csv',
    'transit': processed_path / 'transit_access_by_tract.csv',
    'environment': processed_path / 'environmental_burden_by_tract.csv',
    'schools': processed_path / 'school_features_by_tract.csv',
    'output': processed_path / 'master_tract_features.csv'
}

In [2]:
# loading each cleaned tract-level dataset
acs = pd.read_csv(paths['acs'])
crime = pd.read_csv(paths['crime'])
walkability = pd.read_csv(paths['walkability'])
transit = pd.read_csv(paths['transit'])
environment = pd.read_csv(paths['environment'])
schools = pd.read_csv(paths['schools'])

datasets = {
    'acs': acs,
    'crime': crime,
    'walkability': walkability,
    'transit': transit,
    'environment': environment,
    'schools': schools}

In [3]:
# checking rows and columns before merging
for name, df in datasets.items():
    print(f'{name}: {df.shape[0]} rows, {df.shape[1]} columns')

acs: 737 rows, 50 columns
crime: 737 rows, 13 columns
walkability: 628 rows, 16 columns
transit: 737 rows, 7 columns
environment: 736 rows, 109 columns
schools: 737 rows, 27 columns


In [4]:
# looking for tract ID columns in each dataset
for name, df in datasets.items():
    tract_cols = [
        col for col in df.columns
        if 'tract' in col.lower() or 'geoid' in col.lower() or 'geo_id' in col.lower()]

    print(f'\n{name.upper()}')
    print(tract_cols)


ACS
['geo_id', 'tract_name', 'tract_type_flag']

CRIME
['geo_id', 'tract_name']

WALKABILITY
['tract_geoid']

TRANSIT
['GEOID', 'tract_area_sq_mile', 'large_tract_flag']

ENVIRONMENT
['census_tract', 'census_tract_fips_code', 'tract_fips']

SCHOOLS
['GEOID', 'tract_area_sq_mile']


In [5]:
# checking possible tract ID values and data types
id_checks = {
    'acs': 'geo_id',
    'crime': 'geo_id',
    'walkability': 'tract_geoid',
    'transit': 'GEOID',
    'environment_census_tract': 'census_tract',
    'environment_fips_code': 'census_tract_fips_code',
    'environment_tract_fips': 'tract_fips',
    'schools': 'GEOID'}

for name, col in id_checks.items():
    if name.startswith('environment'):
        df = environment
    else:
        df_name = name
        df = datasets[df_name]

    print(f'\n{name.upper()} - {col}')
    print('dtype:', df[col].dtype)
    print(df[col].head(10).tolist())



ACS - geo_id
dtype: str
['1400000US06073000100', '1400000US06073000201', '1400000US06073000202', '1400000US06073000301', '1400000US06073000302', '1400000US06073000400', '1400000US06073000500', '1400000US06073000600', '1400000US06073000700', '1400000US06073000800']

CRIME - geo_id
dtype: int64
[6073000100, 6073000201, 6073000202, 6073000301, 6073000302, 6073000400, 6073000500, 6073000600, 6073000700, 6073000800]

WALKABILITY - tract_geoid
dtype: int64
[6073000100, 6073000201, 6073000202, 6073000300, 6073000400, 6073000500, 6073000600, 6073000700, 6073000800, 6073000900]

TRANSIT - GEOID
dtype: int64
[6073008331, 6073008336, 6073008337, 6073011601, 6073011602, 6073011700, 6073011801, 6073013307, 6073013308, 6073013309]

ENVIRONMENT_CENSUS_TRACT - census_tract
dtype: int64
[100, 201, 202, 301, 302, 400, 500, 600, 700, 800]

ENVIRONMENT_FIPS_CODE - census_tract_fips_code
dtype: int64
[6073000100, 6073000201, 6073000202, 6073000301, 6073000302, 6073000400, 6073000500, 6073000600, 607300070

In [6]:
# creating one tract ID column for each dataset
acs['tract_id'] = acs['geo_id'].astype(str).str.replace('1400000US', '', regex=False)

crime['tract_id'] = crime['geo_id'].astype(str).str.zfill(11)
walkability['tract_id'] = walkability['tract_geoid'].astype(str).str.zfill(11)
transit['tract_id'] = transit['GEOID'].astype(str).str.zfill(11)
environment['tract_id'] = environment['census_tract_fips_code'].astype(str).str.zfill(11)
schools['tract_id'] = schools['GEOID'].astype(str).str.zfill(11)

In [7]:
# checking tract ID format in each dataset
for name, df in datasets.items():
    tract_id_lengths = df['tract_id'].str.len().unique()
    tract_count = df['tract_id'].nunique()
    row_count = len(df)

    print(f'{name.upper()}')
    print(f'tract id length: {tract_id_lengths}')
    print(f'number of rows: {row_count}')
    print()

ACS
tract id length: [11]
number of rows: 737

CRIME
tract id length: [11]
number of rows: 737

WALKABILITY
tract id length: [11]
number of rows: 628

TRANSIT
tract id length: [11]
number of rows: 737

ENVIRONMENT
tract id length: [11]
number of rows: 736

SCHOOLS
tract id length: [11]
number of rows: 737



Going to start merging by setting ACS as the "master" table for the tracts, the going to merge on tract ID.

In [8]:
# starting with ACS as the base table
master = acs.copy()

print(f'master rows before merge: {master.shape[0]}')
print(f'master columns before merge: {master.shape[1]}')


master rows before merge: 737
master columns before merge: 51


In [9]:
# merging crime features onto ACS base
master = master.merge(
    crime.drop(columns=['geo_id', 'tract_name'], errors='ignore'),
    on='tract_id',
    how='left')

print(f'master rows after crime merge: {master.shape[0]}')
print(f'master columns after crime merge: {master.shape[1]}')


master rows after crime merge: 737
master columns after crime merge: 62


In [10]:
# merging walkability features
master = master.merge(
    walkability.drop(columns=['tract_geoid'], errors='ignore'),
    on='tract_id',
    how='left')

print(f'master rows after walkability merge: {master.shape[0]}')
print(f'master columns after walkability merge: {master.shape[1]}')

master rows after walkability merge: 737
master columns after walkability merge: 77


In [11]:
# merging transit features
master = master.merge(
    transit.drop(columns=['GEOID'], errors='ignore'),
    on='tract_id',
    how='left')

print(f'master rows after transit merge: {master.shape[0]}')
print(f'master columns after transit merge: {master.shape[1]}')

master rows after transit merge: 737
master columns after transit merge: 83


In [12]:
# merging environmental burden features
master = master.merge(
    environment.drop(columns=['census_tract', 'census_tract_fips_code', 'tract_fips'], errors='ignore'),
    on='tract_id',
    how='left')

print(f'master rows after environment merge: {master.shape[0]}')
print(f'master columns after environment merge: {master.shape[1]}')

master rows after environment merge: 737
master columns after environment merge: 189


In [13]:
# checking school columns before merging
schools.columns.tolist()

['GEOID',
 'NAMELSAD',
 'school_count',
 'elementary_school_count',
 'middle_school_count',
 'high_school_count',
 'other_school_count',
 'charter_school_count',
 'tract_area_sq_mile',
 'schools_per_sq_mile',
 'district_name',
 'district_type',
 'district_grade_low',
 'district_grade_high',
 'district_overlap_pct',
 'ela_students_tested',
 'ela_current_status',
 'ela_status_level',
 'ela_change',
 'ela_change_level',
 'math_students_tested',
 'math_current_status',
 'math_status_level',
 'math_change',
 'math_change_level',
 'academic_strength_score',
 'academic_strength_tier',
 'tract_id']

In [14]:
# checking final row and column count
print(f'master rows: {master.shape[0]}')
print(f'master columns: {master.shape[1]}')

master rows: 737
master columns: 189


In [15]:
# merging school features
master = master.merge(
    schools.drop(columns=['GEOID'], errors='ignore'),
    on='tract_id',
    how='left')

print(f'master rows after schools merge: {master.shape[0]}')
print(f'master columns after schools merge: {master.shape[1]}')

master rows after schools merge: 737
master columns after schools merge: 215


In [16]:
# checking final row count and duplicate tract IDs
print(f'master rows: {master.shape[0]}')
print(f'master columns: {master.shape[1]}')
print(f'duplicated tract IDs: {master["tract_id"].duplicated().sum()}')
print(f'unique tracts: {master["tract_id"].nunique()}')

master rows: 737
master columns: 215
duplicated tract IDs: 0
unique tracts: 737


In [17]:
master.shape

(737, 215)

## Check missing values

The merge worked, so now I’ll check missing values.

There are some missing values, like walkability has fewer tracts than ACS, so some walkability fields might be blank. I don’t want to fill anything automatically until I'm clear on what’s missing.

In [18]:
# checking columns with missing values
missing_summary = (
    master.isna()
    .sum() # get a sum of all the missing values
    .reset_index()
    .rename(columns={'index': 'column', 0: 'missing_count'}))

missing_summary['missing_pct'] = (
    missing_summary['missing_count'] / len(master) * 100).round(2)

missing_summary = missing_summary[missing_summary['missing_count'] > 0]

missing_summary.sort_values('missing_count', ascending=False).head(50)

,column,missing_count,missing_pct
176,tsunami_hazard_type_risk_index_score,653,88.60
169,tsunami_exposure_population,653,88.60
175,tsunami_expected_annual_loss_rate_national_per...,653,88.60
168,tsunami_exposure_building_value,653,88.60
167,tsunami_annualized_frequency,653,88.60
171,tsunami_expected_annual_loss_total,653,88.60
172,tsunami_expected_annual_loss_score,653,88.60
170,tsunami_expected_annual_loss_building_value,653,88.60
174,tsunami_expected_annual_loss_rate_building,653,88.60
67,walk_land_acres,201,27.27


After looking into this more closely, I've decided that 700+ tracts is a lot to look at individually, and the scope for looking at various investment opportunities could be too broad.

I want to narrow down the project to look at development projects that are suitable for families. After suburban sprawl, the rising trends were that people would flock to cities and there would be more single people, therefore, development in the 2000s were focused on urban high-rises suitable for 1-2 people, amenities, and proximity to work. However, there have been more recent discussions that those options are no longer serving the people who are having families later.

My goal new goal is to find safe neighborhoods, with access to schools, where there's an opportunity for family development.

## Preview merged dataset

The merge is done, but I’m not filling missing values yet.

First I want to preview the merged table, check the shape, and save the full merged version as processed data.

In [19]:
# previewing the merged dataset
print(f'master rows: {master.shape[0]}')
print(f'master columns: {master.shape[1]}')

master.head()

master rows: 737
master columns: 215


,geo_id,tract_name,total_population_x,population_under_5,population_under_5_rate,population_under_18,population_under_18_rate,population_65_plus,median_age,hispanic_latino_population,hispanic_latino_rate,total_households,households_with_children,households_with_seniors,avg_household_size,avg_family_size,school_enrolled_population,elementary_school_enrollment,high_school_enrollment,bachelors_or_higher,median_household_income,mean_household_income,poverty_rate,family_poverty_rate,employed_population,unemployed_population,unemployment_rate,drove_alone_count,drove_alone_rate,public_transit_commute_count,public_transit_commute_rate,work_from_home_count,work_from_home_rate,total_housing_units,occupied_housing_units,vacant_housing_units,vacancy_rate,owner_occupied_units,renter_occupied_units,renter_rate,vehicle_households,no_vehicle_households,no_vehicle_rate,one_vehicle_households,two_vehicle_households,three_plus_vehicle_households,median_gross_rent,rent_burden_30_34_count,rent_burden_35_plus_count,tract_type_flag,tract_id,total_population_y,total_crime_count,violent_crime_count,property_crime_count,crime_rate_per_1000,violent_crime_rate_per_1000,property_crime_rate_per_1000,violent_safety_score,property_safety_score,safety_score,low_population_flag,walk_block_group_count,walk_tot_pop,walk_households,walk_housing_units,walk_workers,walk_land_acres,nat_walk_index,d2a_jobs_housing_mix,d2b_employment_mix,d3b_intersection_density,d4a_commute_mode_split,d2a_ranked,d2b_ranked,...,heat_wave_expected_annual_loss_rating,heat_wave_expected_annual_loss_rate_building,heat_wave_expected_annual_loss_rate_national_percentile,heat_wave_hazard_type_risk_index_score,heat_wave_hazard_type_risk_index_rating,landslide_annualized_frequency,landslide_exposure_building_value,landslide_exposure_population,landslide_expected_annual_loss_building_value,landslide_expected_annual_loss_total,landslide_expected_annual_loss_score,landslide_expected_annual_loss_rating,landslide_expected_annual_loss_rate_building,landslide_expected_annual_loss_rate_national_percentile,landslide_hazard_type_risk_index_score,landslide_hazard_type_risk_index_rating,inland_flooding_annualized_frequency,inland_flooding_exposure_building_value,inland_flooding_exposure_population,inland_flooding_expected_annual_loss_building_value,inland_flooding_expected_annual_loss_total,inland_flooding_expected_annual_loss_score,inland_flooding_expected_annual_loss_rating,inland_flooding_expected_annual_loss_rate_building,inland_flooding_expected_annual_loss_rate_national_percentile,inland_flooding_hazard_type_risk_index_score,inland_flooding_hazard_type_risk_index_rating,tsunami_annualized_frequency,tsunami_exposure_building_value,tsunami_exposure_population,tsunami_expected_annual_loss_building_value,tsunami_expected_annual_loss_total,tsunami_expected_annual_loss_score,tsunami_expected_annual_loss_rating,tsunami_expected_annual_loss_rate_building,tsunami_expected_annual_loss_rate_national_percentile,tsunami_hazard_type_risk_index_score,tsunami_hazard_type_risk_index_rating,wildfire_annualized_frequency,wildfire_exposure_building_value,wildfire_exposure_population,wildfire_expected_annual_loss_building_value,wildfire_expected_annual_loss_total,wildfire_expected_annual_loss_score,wildfire_expected_annual_loss_rating,wildfire_expected_annual_loss_rate_building,wildfire_expected_annual_loss_rate_national_percentile,wildfire_hazard_type_risk_index_score,wildfire_hazard_type_risk_index_rating,NAMELSAD_y,school_count,elementary_school_count,middle_school_count,high_school_count,other_school_count,charter_school_count,tract_area_sq_mile_y,schools_per_sq_mile,district_name,district_type,district_grade_low,district_grade_high,district_overlap_pct,ela_students_tested,ela_current_status,ela_status_level,ela_change,ela_change_level,math_students_tested,math_current_status,math_status_level,math_change,math_change_level,academic_strength_score,academic_strength_tier
0,1400000US06073000100,Census Tract 1; San Diego

In [20]:
# checking all columns in the merged dataset
master.columns.tolist()

['geo_id',
 'tract_name',
 'total_population_x',
 'population_under_5',
 'population_under_5_rate',
 'population_under_18',
 'population_under_18_rate',
 'population_65_plus',
 'median_age',
 'hispanic_latino_population',
 'hispanic_latino_rate',
 'total_households',
 'households_with_children',
 'households_with_seniors',
 'avg_household_size',
 'avg_family_size',
 'school_enrolled_population',
 'elementary_school_enrollment',
 'high_school_enrollment',
 'bachelors_or_higher',
 'median_household_income',
 'mean_household_income',
 'poverty_rate',
 'family_poverty_rate',
 'employed_population',
 'unemployed_population',
 'unemployment_rate',
 'drove_alone_count',
 'drove_alone_rate',
 'public_transit_commute_count',
 'public_transit_commute_rate',
 'work_from_home_count',
 'work_from_home_rate',
 'total_housing_units',
 'occupied_housing_units',
 'vacant_housing_units',
 'vacancy_rate',
 'owner_occupied_units',
 'renter_occupied_units',
 'renter_rate',
 'vehicle_households',
 'no_vehic

In [21]:
# saving the merged dataset without filling missing values
master.to_csv(paths['output'], index=False)

print(f'saved to: {paths["output"]}')

saved to: C:\Users\cococ\Desktop\Data Science Projects\capstone-3\data\processed\master_tract_features.csv


In [22]:
# columns used directly for scoring
scoring_groups = {
    'safety': [
        'safety_score',
        'violent_safety_score',
        'property_safety_score'
    ],

    'walkability_access': [
        'nat_walk_index',
        'd2a_ranked',
        'd2b_ranked',
        'd3b_ranked',
        'd4a_ranked',
        'transit_stops_per_sq_mile',
        'has_transit_access',
        'public_transit_commute_rate',
        'no_vehicle_rate'
    ],

    'environment_resilience': [
        'expected_annual_loss_score_composite',
        'social_vulnerability_score',
        'community_resilience_score',
        'heat_wave_hazard_type_risk_index_score',
        'inland_flooding_hazard_type_risk_index_score',
        'wildfire_hazard_type_risk_index_score'
    ],

    'schools': [
        'schools_per_sq_mile',
        'academic_strength_score'
    ]
}

In [23]:
# columns kept for EDA, context, filtering, and interpretation
eda_context_groups = {
    'tract_identity': [
        'tract_id',
        'tract_name',
        'total_population_x',
        'tract_type_flag'
    ],

    'demographic_context': [
        'population_under_5_rate',
        'population_under_18_rate',
        'median_age',
        'hispanic_latino_rate',
        'total_households',
        'households_with_children',
        'avg_household_size',
        'avg_family_size',
        'bachelors_or_higher'
    ],

    'economic_housing_context': [
        'median_household_income',
        'poverty_rate',
        'family_poverty_rate',
        'unemployment_rate',
        'drove_alone_rate',
        'work_from_home_rate',
        'renter_rate',
        'median_gross_rent',
        'rent_burden_30_34_count',
        'rent_burden_35_plus_count',
        'vacancy_rate'
    ],

    'crime_context': [
        'total_crime_count',
        'violent_crime_count',
        'property_crime_count',
        'crime_rate_per_1000',
        'violent_crime_rate_per_1000',
        'property_crime_rate_per_1000',
        'low_population_flag'
    ],

    'transit_context': [
        'transit_stop_count',
        'large_tract_flag'
    ],

    'environment_context': [
        'expected_annual_loss_rating_composite',
        'social_vulnerability_rating',
        'community_resilience_rating',
        'heat_wave_hazard_type_risk_index_rating',
        'inland_flooding_hazard_type_risk_index_rating',
        'wildfire_hazard_type_risk_index_rating'
    ],

    'school_context': [
        'school_count',
        'elementary_school_count',
        'middle_school_count',
        'high_school_count',
        'charter_school_count',
        'district_name',
        'district_type',
        'district_grade_low',
        'district_grade_high',
        'district_overlap_pct',
        'academic_strength_tier'
    ]
}

In [24]:
# flattening the groups so they can be used in the dataframe
scoring_cols = [
    col
    for group in scoring_groups.values()
    for col in group
]

eda_context_cols = [
    col
    for group in eda_context_groups.values()
    for col in group
]

mvp_keep_cols = scoring_cols + eda_context_cols

In [25]:
# keeping only columns that actually exist in the merged dataset
mvp_keep_cols = [col for col in mvp_keep_cols if col in master.columns]

mvp_features = master[mvp_keep_cols].copy()

mvp_features.shape

(737, 70)

In [26]:
# columns removed from the MVP dataset
removed_cols = [
    col for col in master.columns
    if col not in mvp_keep_cols
]

removed_cols

['geo_id',
 'population_under_5',
 'population_under_18',
 'population_65_plus',
 'hispanic_latino_population',
 'households_with_seniors',
 'school_enrolled_population',
 'elementary_school_enrollment',
 'high_school_enrollment',
 'mean_household_income',
 'employed_population',
 'unemployed_population',
 'drove_alone_count',
 'public_transit_commute_count',
 'work_from_home_count',
 'total_housing_units',
 'occupied_housing_units',
 'vacant_housing_units',
 'owner_occupied_units',
 'renter_occupied_units',
 'vehicle_households',
 'no_vehicle_households',
 'one_vehicle_households',
 'two_vehicle_households',
 'three_plus_vehicle_households',
 'total_population_y',
 'walk_block_group_count',
 'walk_tot_pop',
 'walk_households',
 'walk_housing_units',
 'walk_workers',
 'walk_land_acres',
 'd2a_jobs_housing_mix',
 'd2b_employment_mix',
 'd3b_intersection_density',
 'd4a_commute_mode_split',
 'NAMELSAD_x',
 'tract_area_sq_mile_x',
 'state_name',
 'state_name_abbreviation',
 'county_name',

In [28]:
rename_cols = {
    'total_population_x': 'total_population',
    'nat_walk_index': 'walkability_index',
    'd2a_ranked': 'jobs_housing_mix_score',
    'd2b_ranked': 'employment_mix_score',
    'd3b_ranked': 'intersection_density_score',
    'd4a_ranked': 'commute_mode_diversity_score',
    'transit_stops_per_sq_mile': 'transit_stop_density',
    'expected_annual_loss_score_composite': 'climate_loss_risk_score',
    'social_vulnerability_score': 'social_vulnerability_score',
    'community_resilience_score': 'community_resilience_score',
    'heat_wave_hazard_type_risk_index_score': 'heat_risk_score',
    'inland_flooding_hazard_type_risk_index_score': 'flood_risk_score',
    'wildfire_hazard_type_risk_index_score': 'wildfire_risk_score',
    'schools_per_sq_mile': 'school_density',
    'academic_strength_score': 'school_academic_score'}

mvp_features = mvp_features.rename(columns=rename_cols)

## Cleaning irrelevant features

From 215 to 70 features.

The full merged dataset had over 200 columns, so I created a smaller MVP dataset for EDA and scoring.

I kept columns that were the most straight forward for safety, walkability, transit access, environmental risk, school access, and basic neighborhood context. For scoring I kept rates over counts, since some tracts are larger or more populated than others. For example, `transit_stops_per_sq_mile` is more useful than just `transit_stop_count` because larger tracts can naturally have more stops.

I kept some demographic, income, housing, and school district fields for EDA/context, so I have some idea of what kind of neighborhood each tract represents, but they might not play into the final scoring system.

Removed columns: most raw count fields, duplicate merge columns, and very detailed hazard fields. If I were doing a larger scope or a deep analysis, I'd keep them, but for this version, I’m using a smaller set of clear features.

Renaming: Some of the column names were unclear, so I renamed them to something easier to interpret.